# Clean, Source-Organized Copy
Saved outputs and machine-specific metadata are intentionally excluded. The byte-identical original remains in the local audit archive. Prepare the local workbench before running this notebook. Notebooks 01 and 02 execute GPU inference when explicitly enabled; 03 disables its optional GPU activation check by default.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts" / "lab.py").is_file()
)
WORKSPACE = PROJECT_ROOT / ".local" / "workbench"
if not (WORKSPACE / "weight_edit.py").is_file():
    raise RuntimeError("Run python scripts/lab.py prepare --with-local-data first.")
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))


# Does the prompt encourage unnecessary rechecking?

The previous weight-edit pilot did not improve delivered answers. This is a **new diagnostic question**, not a reopened weight sweep: compare thinking on/off against step-by-step/final-only instructions, using only the original model.

Four conditions x two existing development questions x two seed groups = **16 responses**. No weights are edited, no new problems are selected, and the held-out set stays unused.

Select **Python (weight-space)** and run all cells. By default this notebook reads the completed local report without rerunning inference.

In [ ]:
from pathlib import Path
from statistics import mean

import torch

from experiment_runner import load_local_model, run_suite
from math_probe import PROMPT_INSTRUCTIONS
from prompt_control import (
    DESIGN_VERSION, PROMPT_CONDITIONS, PROMPT_PROBLEMS, PROMPT_SEEDS,
    load_prompt_report,
)
from weight_edit import PROJECT_ROOT

RUN_EXPERIMENT = False
MAX_NEW_TOKENS = 1024
WALL_BUDGET_SECONDS = 900
TOTAL_TOKEN_BUDGET = 16384
REPORT_PATH = None

## The four controls

| Condition | Thinking | Instruction |
| --- | --- | --- |
| think_on_steps | On | Reason step by step |
| think_on_final | On | Final boxed answer only |
| think_off_steps | Off | Reason step by step |
| think_off_final | Off | Final boxed answer only |

The same question and per-question seed are paired across all four conditions. Temperature, top-p, top-k, presence penalty, token cap, and grading stay fixed. Thinking-on can still generate internal reasoning even when the requested final response is concise.

The existing step-by-step wording is unchanged. The full user message, rendered chat prompt, settings, and source hashes are recorded. Answers used for scoring are never added to the model's prompt.

In [ ]:
print('Design:', DESIGN_VERSION, '| base checkpoint only')
for condition in PROMPT_CONDITIONS:
    print(condition.name, '| thinking:', condition.thinking, '| style:', condition.prompt_style)
for style, instruction in PROMPT_INSTRUCTIONS.items():
    print('\n' + style + ':', instruction)
print('\nDevelopment question IDs:', [problem['id'] for problem in PROMPT_PROBLEMS])
print('Seed groups:', PROMPT_SEEDS, '(actual per-question seed = group + question index)')
print('Soft wall-time limit:', WALL_BUDGET_SECONDS, 'seconds')
print('Generation-token allocation:', TOTAL_TOKEN_BUDGET)

In [ ]:
if type(RUN_EXPERIMENT) is not bool:
    raise ValueError('RUN_EXPERIMENT must be True or False.')
if RUN_EXPERIMENT:
    model, tokenizer = load_local_model()
    try:
        run_dir, report = run_suite(
            model, tokenizer, stage='prompt_control',
            problems=PROMPT_PROBLEMS, conditions=PROMPT_CONDITIONS, seeds=PROMPT_SEEDS,
            max_new_tokens=MAX_NEW_TOKENS, seconds=WALL_BUDGET_SECONDS,
            token_budget=TOTAL_TOKEN_BUDGET,
        )
        REPORT_PATH = run_dir / 'summary.json'
    finally:
        del model
        torch.cuda.empty_cache()
else:
    print('GPU experiment skipped. Reading a saved local report only.')

In [ ]:
if REPORT_PATH is None:
    candidates = list((PROJECT_ROOT / 'results').glob('prompt_control_[0-9]*/summary.json'))
    if not candidates:
        raise FileNotFoundError('No saved prompt-control report yet. Wait for the running experiment, or enable RUN_EXPERIMENT.')
    REPORT_PATH = max(candidates, key=lambda path: path.stat().st_mtime)
REPORT_PATH = Path(REPORT_PATH)
if not REPORT_PATH.is_absolute():
    REPORT_PATH = PROJECT_ROOT / REPORT_PATH
report = load_prompt_report(REPORT_PATH)
print('Report:', REPORT_PATH.relative_to(PROJECT_ROOT))
print('Status:', report['status'], '| generations:', report['completed_generations'], '/', report['expected_generations'])
print('Base-only plan:', report['base_only'])
print('Original target tensor hashes verified:', report['restored_base_hashes_verified'])
print('Generated tokens:', report['used_tokens'], '| wall seconds:', round(report['wall_seconds'], 1))

## Delivery, stopping, and cost

**Delivered correct** means the final answer is correctly boxed and the model emits a natural EOS stop. A correct answer inside unfinished thinking does not count as delivered. Natural stopping and token caps are reported separately.

A shorter response is not automatically an improvement: inspect correctness and completion alongside token use. The final-only instruction is an experimental input, not a guarantee of format compliance. Repeated four-token grams are only a heuristic, and local timings are not speed benchmarks.

In [ ]:
print(f"{'condition':<19} {'delivered':<12} {'EOS':<8} {'cap':<8} {'tokens':<9} {'repeat4':<9} {'sec':<8}")
for condition in PROMPT_CONDITIONS:
    row = report['summary'].get(condition.name)
    if row is None:
        continue
    records = [item for item in report['records'] if item['condition'] == condition.name]
    repeat = mean(item['repeated_4gram_fraction'] for item in records)
    delivered = str(row['delivered_correct']) + '/' + str(row['count'])
    print(f"{condition.name:<19} {delivered:<12} {row['natural_stop_rate']:<8.0%} "
          f"{row['cap_rate']:<8.0%} {row['mean_tokens']:<9.1f} {repeat:<9.1%} {row['mean_seconds']:<8.1f}")

In [ ]:
decision = report['decision']
print(decision['reason'])
print('\nDeltas are treatment minus reference. Negative token delta means fewer generated tokens.')
for effect in decision['comparisons']:
    print('\n' + effect['name'] + ':', effect['reference'], '->', effect['treatment'])
    print('  Delivered-correct wins/losses/ties:', effect['delivered_correct_wins'],
          effect['delivered_correct_losses'], effect['delivered_correct_ties'], 'over', effect['pairs'], 'pairs')
    print(f"  Delivery delta: {effect['delivered_correct_rate_delta']:+.0%}; "
          f"natural-stop delta: {effect['natural_stop_rate_delta']:+.0%}; "
          f"mean token delta: {effect['mean_token_delta']:+.1f}")
for seed, table in decision['by_seed_group'].items():
    print('\nSeed group', seed)
    for name, row in table.items():
        print(' ', name, '| delivered:', str(row['delivered_correct']) + '/' + str(row['count']))

In [ ]:
for condition in PROMPT_CONDITIONS:
    sample = next((row for row in report['records']
                   if row['condition'] == condition.name and row['problem_id'] == 'discount'
                   and row['seed_group'] == 0), None)
    if sample:
        print('\n---', condition.name, '/ discount / seed group 0 ---')
        print('Stop:', sample['stop_reason'], '| delivered correct:', sample['delivered_correct'])
        print(sample['text'][:1500])

## Stop and interpret

This design ends after 16 responses or the declared budget. Partial, unpaired, or time-limited runs produce no paired effect estimates. Two questions and two seed groups do **not** establish general math accuracy or four independent tasks.

If a concise instruction helps, that is a prompt-level finding, not evidence of a successful weight edit. If it does not help, a separate reference-protocol comparison needs an explicit hypothesis and budget. Do not silently extend this sweep or consume the held-out set.

The earlier weight-edit gate stays closed. This experiment cannot authorize held-out inference.